In [346]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import talib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import os
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau 
import shap
import plotly.graph_objs as go
import plotly.offline as pyo
from tqdm.auto import tqdm


In [347]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("gpu")
else:
    device = torch.device('cpu')
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())

gpu
2.1.2+cu121
CUDA available: True
CUDA version: 12.1
cuDNN version: 8902


In [348]:
start_date = '2018-01-01'
end_date = '2024-01-24'

stock_data = yf.download("AAPL", start=start_date, end=end_date)[["Adj Close", "High", "Low", "Volume"]]

stock_data = stock_data.reset_index()

stock_data = stock_data[['Date', 'Adj Close', "High", "Low", "Volume"]]

stock_data = stock_data.sort_values(by="Date")
stock_data.head(45)

[*********************100%%**********************]  1 of 1 completed


,Date,Adj Close,High,Low,Volume
0,2018-01-02,40.722874,43.075001,42.314999,102223600
1,2018-01-03,40.715782,43.637501,42.990002,118071600
2,2018-01-04,40.904903,43.367500,43.020000,89738400
3,2018-01-05,41.370617,43.842499,43.262501,94640000
4,2018-01-08,41.216972,43.902500,43.482498,82271200
5,2018-01-09,41.212234,43.764999,43.352501,86336000
6,2018-01-10,41.202774,43.575001,43.250000,95839600
7,2018-01-11,41.436810,43.872501,43.622501,74670800
8,2018-01-12,41.864704,44.340000,43.912498,101672400
9,2018-01-16,41.651939,44.847500,44.035000,118263600


In [349]:
time_step = 44

In [350]:
test_index = int((len(stock_data)-44)*0.8+44+44)

In [351]:
date = stock_data["Date"].iloc[time_step:].dt.strftime('%Y-%m-%d')
date_test = stock_data["Date"].iloc[test_index:].reset_index()
date_test.drop(columns=["index"], inplace=True)
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [352]:
def add_technical_indicators(data, timeperiod=time_step):

    # MACD
    macd, macdsignal, macdhist = talib.MACD(data["Adj Close"], fastperiod=12, slowperiod=26, signalperiod=9)

    rsi = talib.RSI(data["Adj Close"], timeperiod=14)

    # CMO
    cmo = talib.CMO(data["Adj Close"], timeperiod=timeperiod)

    # MOM
    mom = talib.MOM(data["Adj Close"], timeperiod=timeperiod)

    # Bollinger Bands
    upperband, middleband, lowerband = talib.BBANDS(data["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

    # SMA
    sma = talib.SMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Exponential Moving Average (EMA)
    ema = talib.EMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Stochastic Oscillator
    slowk, slowd = talib.STOCH(data['High'], data['Low'], data['Adj Close'], fastk_period=14, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

    # Calculate Average True Range (ATR)
    atr = talib.ATR(data['High'], data['Low'], data['Adj Close'], timeperiod=14)

    # Calculate On-Balance Volume (OBV)
    obv = talib.OBV(data['Adj Close'], data['Volume'])

    # Combine all indicators into a DataFrame
    indicators = pd.DataFrame({
        'MACD': macd,
        'MACD_signal': macdsignal,
        'RSI': rsi,
        'CMO': cmo,
        'MOM': mom,
        'Upper_BB': upperband,
        'Middle_BB': middleband,
        'Lower_BB': lowerband,
        'SMA': sma,
        'EMA': ema,
        'SLOWK': slowk,
        'SLOWD': slowd,
        'ATR': atr,
        'OBV': obv,

    })
    return indicators

In [353]:
indicators = add_technical_indicators(stock_data)
indicators.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0


In [354]:
indicators_with_price = pd.concat([indicators, stock_data["Adj Close"]], axis=1, join='inner')
indicators_with_price.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0,40.722874
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0,40.715782
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0,40.904903
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0,41.370617
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0,41.216972
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0,41.212234
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0,41.202774
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0,41.436810
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0,41.864704
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0,41.651939


In [355]:
indicators_with_price = indicators_with_price.dropna()
indicators_with_price = indicators_with_price.reset_index(drop=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.420518,0.238064,56.052942,4.051391,0.823544,44.028283,40.539883,37.051484,40.615035,40.638545,12.020130,29.597831,2.704890,-4.208332e+08,41.546417
1,0.431503,0.276752,59.233514,6.192289,1.284008,44.042944,40.754081,37.465219,40.644217,40.699045,-9.379546,12.287852,2.706939,-3.257368e+08,41.999790
2,0.492755,0.319952,63.732450,9.481636,1.816471,43.867403,41.056249,38.245095,40.685501,40.788926,-18.948476,-5.435964,2.727887,-1.969960e+08,42.721375
3,0.568076,0.369577,66.042453,11.303216,1.763779,43.662506,41.356637,39.050768,40.725586,40.893169,-6.400755,-11.576259,2.738476,-6.816760e+07,43.134396
4,0.587478,0.413157,61.780485,9.044919,1.502037,43.567676,41.561485,39.555293,40.759724,40.974318,1.684805,-7.888142,2.738628,-1.949416e+08,42.719009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [356]:
indicators_with_price['Next_Adj_Close'] = indicators_with_price['Adj Close'].shift(-1)
indicators_with_price['Return'] = ((indicators_with_price['Next_Adj_Close'] - indicators_with_price['Adj Close'])/indicators_with_price['Adj Close'])*100
indicators_with_price['Signal'] = np.where(indicators_with_price['Return'] > 1, 1,
                                           np.where(indicators_with_price['Return'] < -1, 2, 0))
indicators_with_price


,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Next_Adj_Close,Return,Signal
0,0.420518,0.238064,56.052942,4.051391,0.823544,44.028283,40.539883,37.051484,40.615035,40.638545,12.020130,29.597831,2.704890,-4.208332e+08,41.546417,41.999790,1.091244,1
1,0.431503,0.276752,59.233514,6.192289,1.284008,44.042944,40.754081,37.465219,40.644217,40.699045,-9.379546,12.287852,2.706939,-3.257368e+08,41.999790,42.721375,1.718066,1
2,0.492755,0.319952,63.732450,9.481636,1.816471,43.867403,41.056249,38.245095,40.685501,40.788926,-18.948476,-5.435964,2.727887,-1.969960e+08,42.721375,43.134396,0.966779,0
3,0.568076,0.369577,66.042453,11.303216,1.763779,43.662506,41.356637,39.050768,40.725586,40.893169,-6.400755,-11.576259,2.738476,-6.816760e+07,43.134396,42.719009,-0.963005,0
4,0.587478,0.413157,61.780485,9.044919,1.502037,43.567676,41.561485,39.555293,40.759724,40.974318,1.684805,-7.888142,2.738628,-1.949416e+08,42.719009,42.355839,-0.850138,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,188.630005,3.257068,1
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,191.559998,1.553301,1
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,193.889999,1.216330,1
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,195.179993,0.665322,0


In [357]:
indicators_with_price["Signal"].value_counts()

Signal
0    740
1    416
2    324
Name: count, dtype: int64

In [358]:
indicators_with_price = indicators_with_price.drop(columns=['Next_Adj_Close', 'Return'])
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Signal
0,0.420518,0.238064,56.052942,4.051391,0.823544,44.028283,40.539883,37.051484,40.615035,40.638545,12.020130,29.597831,2.704890,-4.208332e+08,41.546417,1
1,0.431503,0.276752,59.233514,6.192289,1.284008,44.042944,40.754081,37.465219,40.644217,40.699045,-9.379546,12.287852,2.706939,-3.257368e+08,41.999790,1
2,0.492755,0.319952,63.732450,9.481636,1.816471,43.867403,41.056249,38.245095,40.685501,40.788926,-18.948476,-5.435964,2.727887,-1.969960e+08,42.721375,0
3,0.568076,0.369577,66.042453,11.303216,1.763779,43.662506,41.356637,39.050768,40.725586,40.893169,-6.400755,-11.576259,2.738476,-6.816760e+07,43.134396,0
4,0.587478,0.413157,61.780485,9.044919,1.502037,43.567676,41.561485,39.555293,40.759724,40.974318,1.684805,-7.888142,2.738628,-1.949416e+08,42.719009,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,1
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,1
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,1
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,0


In [359]:
indicators_with_price.head(50)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Signal
0,0.420518,0.238064,56.052942,4.051391,0.823544,44.028283,40.539883,37.051484,40.615035,40.638545,12.020130,29.597831,2.704890,-4.208332e+08,41.546417,1
1,0.431503,0.276752,59.233514,6.192289,1.284008,44.042944,40.754081,37.465219,40.644217,40.699045,-9.379546,12.287852,2.706939,-3.257368e+08,41.999790,1
2,0.492755,0.319952,63.732450,9.481636,1.816471,43.867403,41.056249,38.245095,40.685501,40.788926,-18.948476,-5.435964,2.727887,-1.969960e+08,42.721375,0
3,0.568076,0.369577,66.042453,11.303216,1.763779,43.662506,41.356637,39.050768,40.725586,40.893169,-6.400755,-11.576259,2.738476,-6.816760e+07,43.134396,0
4,0.587478,0.413157,61.780485,9.044919,1.502037,43.567676,41.561485,39.555293,40.759724,40.974318,1.684805,-7.888142,2.738628,-1.949416e+08,42.719009,0
5,0.567013,0.443928,58.241621,7.100861,1.143604,43.382885,41.728829,40.074773,40.785715,41.035718,-7.013504,-3.909818,2.715225,-3.124152e+08,42.355839,0
6,0.548495,0.464842,58.592215,7.332888,1.202911,43.261029,41.862705,40.464380,40.813054,41.096606,-20.016654,-8.448451,2.714435,-2.214400e+08,42.405685,0
7,0.515805,0.475034,57.044806,6.516173,0.819328,43.280287,41.922402,40.564517,40.831675,41.148141,-27.991976,-18.340711,2.690141,-3.790588e+08,42.256138,2
8,0.432812,0.466590,50.806350,3.052092,-0.254204,43.245415,41.956465,40.667515,40.825897,41.168690,-36.985499,-28.331377,2.648799,-5.128460e+08,41.610500,0
9,0.361721,0.445616,50.674822,2.976570,-0.055668,43.183913,41.996699,40.809486,40.824632,41.187694,-46.752180,-37.243219,2.644564,-5.914436e+08,41.596272,2


In [360]:
y = indicators_with_price["Adj Close"]
y_2 = indicators_with_price["SMA"]
y_3 = indicators_with_price["EMA"]
y_4 = indicators_with_price["Upper_BB"]
y_5 = indicators_with_price["Middle_BB"]
y_6 = indicators_with_price["Lower_BB"]
X = np.array(date)

trace = go.Scatter(x=X, y=y, mode="lines", name="Adj Close")
trace_2 = go.Scatter(x=X, y=y_2, mode="lines", name="SMA")
trace_3 = go.Scatter(x=X, y=y_3, mode="lines", name="EMA")
trace_4 = go.Scatter(x=X, y=y_4, mode="lines", name="Upper_BB")
trace_5 = go.Scatter(x=X, y=y_5, mode="lines", name="Middle_BB")
trace_6 = go.Scatter(x=X, y=y_6, mode="lines", name="Lower_BB")



layout = go.Layout(
    title='Stock Price and Volume',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Adj Close', side='left', rangemode='tozero'),
    yaxis2=dict(title='SMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis3=dict(title='EMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis4=dict(title='Upper_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis5=dict(title='Middle_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis6=dict(title='Lower_BB', side='right', overlaying='y', rangemode='tozero'),
    height=600,
)

fig = go.Figure(data=[trace, trace_2, trace_3, trace_4, trace_5, trace_6], layout=layout)

# Show plot
pyo.iplot(fig)

In [361]:
class RollingWindowDataset(Dataset):
    def __init__(self, X, y, window_size, device="gpu"):
        self.X = X.clone().detach().to(torch.float).to(device)
        self.y = y.clone().detach().to(torch.float).to(device)
        self.window_size = window_size

    def __len__(self):
        # Adjust the length to account for window size
        return len(self.X) - self.window_size 

    def __getitem__(self, idx):
        # Ensure idx is within the valid range
        if idx + self.window_size > len(self.X):
            raise IndexError("Index out of bounds")

        # Extract a window of data from X
        X_window = self.X[idx : idx + self.window_size]
        
        # Assuming you're predicting the value right after the window
        y_target = self.y[idx + self.window_size]  

        return X_window.clone().detach().to(torch.float), y_target.clone().detach().to(torch.float)


In [362]:
X = indicators_with_price.iloc[:, :-1]
y = indicators_with_price.iloc[:,-1]

signal_true = indicators_with_price.iloc[:,-1]
y

0       1
1       1
2       0
3       0
4       0
       ..
1475    1
1476    1
1477    1
1478    0
1479    0
Name: Signal, Length: 1480, dtype: int64

In [363]:
train_signal_true = signal_true.iloc[:int(len(X)*0.8)]
test_signal_true = signal_true.iloc[int(len(X)*0.8):]
test_signal_true

1184    1
1185    0
1186    2
1187    1
1188    0
       ..
1475    1
1476    1
1477    1
1478    0
1479    0
Name: Signal, Length: 296, dtype: int64

In [364]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# X_train = X_train.to_numpy()
# y_train = y_train.to_numpy()
y_test.head(44)

1184    1
1185    0
1186    2
1187    1
1188    0
1189    2
1190    2
1191    2
1192    1
1193    0
1194    0
1195    0
1196    2
1197    2
1198    1
1199    0
1200    1
1201    0
1202    2
1203    2
1204    2
1205    2
1206    0
1207    1
1208    2
1209    0
1210    2
1211    2
1212    1
1213    0
1214    2
1215    1
1216    2
1217    1
1218    0
1219    0
1220    1
1221    0
1222    1
1223    0
1224    0
1225    0
1226    1
1227    1
Name: Signal, dtype: int64

In [365]:
# # Get the count of each unique value in the series
# category_counts = y_train.value_counts()
# print(category_counts.index)
# print(category_counts.values)

# # Create bar plot
# plt.bar(["1","0"], category_counts.values)

# # Adding labels and title
# plt.xlabel('Category')
# plt.ylabel('Frequency')
# plt.title('Distribution of Categories')

# # Show plot
# plt.show()

In [366]:
X_train_df = pd.DataFrame()
X_test_df = pd.DataFrame()
scaler_dict = {}

for column in X_train.columns:
    # Initialize a scaler
    scaler = MinMaxScaler()
    
    # Fit on the training data and transform it
    X_train_scaled = scaler.fit_transform(X_train[column].to_numpy().reshape(-1, 1))
    X_train_df[column] = X_train_scaled.flatten()
    
    # Transform the test data using the same scaler
    X_test_scaled = scaler.transform(X_test[column].to_numpy().reshape(-1, 1))
    X_test_df[column] = X_test_scaled.flatten()

    scaler_dict[column] = scaler

    

X_train_df.head(24)

features = X_train_df.columns
X_test_df

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.473178,0.391410,0.484405,0.368317,0.443981,0.810260,0.788953,0.749683,0.804887,0.832500,0.850519,0.839325,0.765326,0.761787,0.780636
1,0.499847,0.411144,0.517394,0.387563,0.487569,0.813419,0.791684,0.751875,0.804933,0.833706,0.876575,0.853920,0.764804,0.775662,0.793797
2,0.523637,0.432238,0.527280,0.393264,0.448749,0.816205,0.793223,0.752021,0.804433,0.835055,0.895160,0.867699,0.724522,0.788577,0.797684
3,0.523008,0.448973,0.456306,0.359879,0.379729,0.815520,0.792793,0.751879,0.802963,0.835217,0.907286,0.887937,0.685712,0.778442,0.775317
4,0.534244,0.464868,0.497611,0.382098,0.444492,0.813808,0.792104,0.752318,0.802403,0.836117,0.913913,0.901248,0.661728,0.787383,0.790114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,0.331987,0.360467,0.219193,0.330143,0.438975,1.105702,1.104494,1.078159,1.146372,1.152418,0.696648,0.699736,0.263755,0.912203,1.018693
292,0.358007,0.354748,0.456177,0.434230,0.530971,1.099583,1.101858,1.079371,1.147028,1.152792,0.728273,0.697238,0.316961,0.925666,1.059493
293,0.396417,0.358741,0.541661,0.480564,0.534505,1.093074,1.099905,1.082407,1.147733,1.154161,0.814260,0.731128,0.316624,0.937531,1.079584
294,0.440665,0.371807,0.601114,0.515679,0.555950,1.091863,1.099564,1.083019,1.148741,1.156274,0.926300,0.813002,0.323439,0.947909,1.095561


In [367]:
scaler_dict

{'MACD': MinMaxScaler(),
 'MACD_signal': MinMaxScaler(),
 'RSI': MinMaxScaler(),
 'CMO': MinMaxScaler(),
 'MOM': MinMaxScaler(),
 'Upper_BB': MinMaxScaler(),
 'Middle_BB': MinMaxScaler(),
 'Lower_BB': MinMaxScaler(),
 'SMA': MinMaxScaler(),
 'EMA': MinMaxScaler(),
 'SLOWK': MinMaxScaler(),
 'SLOWD': MinMaxScaler(),
 'ATR': MinMaxScaler(),
 'OBV': MinMaxScaler(),
 'Adj Close': MinMaxScaler()}

In [368]:
correlation_matrix = X_train_df.corr()

# Create the heatmap with text
fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='Viridis',
                    text=correlation_matrix.round(2).values,  # Rounded values for display
                    texttemplate="%{text}",
                    textfont={"size":9}  # Adjust text size if necessary
                    ))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  # or any width you desire
    height=1000,  # or any height you desire
)

# Show the figure
pyo.iplot(fig)

In [369]:
X_train_tensor = torch.tensor(X_train_df.to_numpy(), dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float, device=device)

X_test_tensor = torch.tensor(X_test_df.to_numpy(), dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float, device=device)

train_data = RollingWindowDataset(X_train_tensor, y_train_tensor, window_size=time_step, device=device)
test_data = RollingWindowDataset(X_test_tensor, y_test_tensor, window_size=time_step, device=device)

print(test_data.__getitem__(0)[1])
print(test_data.__getitem__(0)[0])


tensor(1., device='cuda:0')
tensor([[0.4732, 0.3914, 0.4844, 0.3683, 0.4440, 0.8103, 0.7890, 0.7497, 0.8049,
         0.8325, 0.8505, 0.8393, 0.7653, 0.7618, 0.7806],
        [0.4998, 0.4111, 0.5174, 0.3876, 0.4876, 0.8134, 0.7917, 0.7519, 0.8049,
         0.8337, 0.8766, 0.8539, 0.7648, 0.7757, 0.7938],
        [0.5236, 0.4322, 0.5273, 0.3933, 0.4487, 0.8162, 0.7932, 0.7520, 0.8044,
         0.8351, 0.8952, 0.8677, 0.7245, 0.7886, 0.7977],
        [0.5230, 0.4490, 0.4563, 0.3599, 0.3797, 0.8155, 0.7928, 0.7519, 0.8030,
         0.8352, 0.9073, 0.8879, 0.6857, 0.7784, 0.7753],
        [0.5342, 0.4649, 0.4976, 0.3821, 0.4445, 0.8138, 0.7921, 0.7523, 0.8024,
         0.8361, 0.9139, 0.9012, 0.6617, 0.7874, 0.7901],
        [0.5474, 0.4805, 0.5145, 0.3912, 0.4671, 0.8153, 0.7928, 0.7521, 0.8022,
         0.8373, 0.9235, 0.9113, 0.6236, 0.7974, 0.7962],
        [0.5399, 0.4914, 0.4471, 0.3605, 0.4592, 0.8163, 0.7941, 0.7537, 0.8018,
         0.8374, 0.9237, 0.9172, 0.5981, 0.7914, 0.7760],

# Vanilla LSTM

In [370]:
class VanillaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob):
        super(VanillaLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

# Stacked LSTM

In [371]:
class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, stateful):
        super(StackedLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size
        self.stateful = stateful
        
        self.hn = None
        self.cn = None

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=self.layer_size,
                            dropout=(dropout_prob if self.layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def reset_hidden_state(self, batch_size=None):
        if batch_size is None or not self.stateful:
            # Reset hidden and cell states to None if batch_size is not provided or if the model is not stateful
            self.hn = None
            self.cn = None
        else:
            # Resize hidden and cell states to match current batch size, preserving the state as much as possible
            self.hn = self._resize_state(self.hn, batch_size)
            self.cn = self._resize_state(self.cn, batch_size)


    def _resize_state(self, state, batch_size):
        if state is None:
            # If state is not initialized, initialize with zeros
            return torch.zeros(self.layer_size, batch_size, self.hidden_size).to(device)
        elif batch_size < state.size(1):
            # If batch size is smaller, truncate the state
            return state[:, :batch_size, :].contiguous()
        elif batch_size > state.size(1):
            # If batch size is larger, pad the state with zeros
            padding_size = batch_size - state.size(1)
            padding = torch.zeros(self.layer_size, padding_size, self.hidden_size).to(device)
            return torch.cat([state, padding], dim=1)
        

    def forward(self, x):
        current_batch_size = x.size(0)

        # Check and adjust the hidden and cell states
        if not self.stateful or self.hn is None or self.hn.size(1) != current_batch_size:
            self.reset_hidden_state(current_batch_size)
        else:
            # Detach the hidden states from the graph to avoid backpropagating through the entire sequence history
            self.hn = self.hn.detach()
            self.cn = self.cn.detach()

        # Ensure that hn and cn are not None and have the correct shape
        h0 = self.hn if self.hn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)
        c0 = self.cn if self.cn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Update hidden states if stateful
        if self.stateful:
            self.hn, self.cn = hn.detach(), cn.detach()

        # Process the output of the last time step
        out = self.dropout(out[:, -1, :])  # Add dropout
        out = self.fc(out)  # Fully connected layer

        return out

# Bidirectional LSTM

# 1D CNN-LSTM

In [372]:
class OneDimCNNLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, conv_channels, kernel_size, pool_size, stride):
        super(OneDimCNNLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        # Convolutional Layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=conv_channels, kernel_size=kernel_size)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_size)

        # self.conv2 = nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels*2, kernel_size=kernel_size)
        # self.relu2 = nn.ReLU()
        # self.maxpool2 = nn.MaxPool1d(kernel_size=2, stride=2)


        self.lstm = nn.LSTM(input_size = conv_channels, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # CNN in_dim: (batch_size, in_channels, length)
        x = x.transpose(1, 2)

        # Conv layer forward propagate
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        # x = self.conv2(x)
        # x = self.relu2(x)
        # x = self.maxpool2(x)

        # LSTM in_dim: (batch_size, seq_len, input_size)
        x = x.transpose(1, 2)

        assert x.size(-1) == self.lstm.input_size, f"Mismatch in LSTM input size. Expected: {self.lstm.input_size}, Got: {x.size(-1)}"

        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

In [373]:
class ModelActioner:
    
    def __init__(self, train_data, test_data, device, model_type):
        self.train_data = train_data
        self.test_data = test_data
        self.device = device
        self.model_type = model_type
        self.model = None
        self.optimizer = None
        self.criterion = nn.CrossEntropyLoss()

    
    def train_validate(self, config, trial):

        if self.model_type == "Vanilla LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=3).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=3).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=3, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        n_splits = 5
        tscv = TimeSeriesSplit(n_splits=n_splits)

        val_losses = []

        for fold, (train_ids, val_ids) in enumerate(tscv.split(self.train_data)):
            print(f'Starting fold {fold+1}:')

            self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

            scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True) 

            train_subset = Subset(self.train_data, train_ids)
            val_subset = Subset(self.train_data, val_ids)
            
            # Creating data loader
            train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

            # Training Loop
            for epoch in range(epochs):
                print('epochs {}/{}'.format(epoch+1,epochs))

                running_loss = 0.0
                total_sample_train = 0

                self.model.train()

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(self.device), target.to(self.device)
                    target = target.long()


                    self.optimizer.zero_grad()
                    preds = self.model(data)

                    loss = self.criterion(preds, target)
                    loss = loss.mean()
                    loss.backward()
                    self.optimizer.step() # Update model params

                    running_loss += loss.item() * data.size(0)
                    total_sample_train += data.size(0)

                train_loss = running_loss/total_sample_train
                #print(f"train loss: {train_loss}")

                self.model.eval()
                val_running_loss = 0.0
                total_sample_val = 0

                with torch.no_grad():

                    for batch_idx, (data, target) in enumerate(val_loader):
                        data, target = data.to(self.device), target.to(self.device)
                        target = target.long()

                        preds = self.model(data)
                        loss = self.criterion(preds, target)
                        loss = loss.mean()

                        val_running_loss += loss.item() * data.size(0)
                        total_sample_val += data.size(0)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                val_loss = val_running_loss/total_sample_val
                val_losses.append(val_loss)
                scheduler.step(val_loss)
                
                unique_step = fold * epochs + epoch
                trial.report(val_loss, unique_step)

                current_lr = self.optimizer.param_groups[0]['lr']

                print(f'Current Learning Rate: {current_lr}')
                print(f"train_loss: {train_loss}, val_loss: {val_loss}")
                
        mean_val_loss = np.mean(val_losses)
        print(f"Mean validation loss: {mean_val_loss}")
        return mean_val_loss
    
                    
    def train(self, config):
        if self.model_type == "Vanilla LSTM":

            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=3).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=3).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=3, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        # Update optimizer with updated lr
        self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

        # Creating data loader
        train_loader = DataLoader(dataset=self.train_data, batch_size=batch_size, shuffle=False)

        scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True)  

        # Training Loop
        for epoch in range(epochs):
            print('epochs {}/{}'.format(epoch+1,epochs))

            running_loss = 0.0
            total_sample_train = 0

            self.model.train()

            for batch_idx, (data, target) in enumerate(train_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()  

                self.optimizer.zero_grad()
                preds = self.model(data)

                loss = self.criterion(preds, target)
                loss = loss.mean()
                loss.backward()
                self.optimizer.step() # Update model params

                running_loss += loss.item() * data.size(0)
                total_sample_train += data.size(0)

            train_loss = running_loss/total_sample_train
            #print(f"train loss: {train_loss}")
            scheduler.step(train_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            print(f'Current Learning Rate: {current_lr}')
            print(f"train_loss: {train_loss}")
        
        return self.model
            
    
    def test(self, config):
        batch_size = config["batch_size"]
        all_preds = []

        test_loader = DataLoader(dataset=self.test_data, batch_size=batch_size, shuffle=False)

        running_loss = .0
        total_sample = 0

        self.model.eval()

        with torch.no_grad():

            for batch_idx, (data, target) in enumerate(test_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()
                
                preds = self.model(data)
                loss = self.criterion(preds, target)

                running_loss += loss.item() * data.size(0)
                total_sample += data.size(0)

                _, predicted_classes = torch.max(preds, 1)
                all_preds.extend(predicted_classes.cpu().numpy())

            test_loss = running_loss/total_sample
            print(f"test_loss: {test_loss}")

        return all_preds
    


In [374]:
model_type = "1D CNN-LSTM"

def objective(trial):
    if model_type == "Vanilla LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.15),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 10, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9)
        }

    elif model_type == "Stacked LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "num_layers": trial.suggest_int("num_layers", 1, 5)
        }

    elif model_type == "1D CNN-LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "conv_channels": trial.suggest_int("conv_channels", 16, 128),
            "kernel_size": trial.suggest_int("kernel_size", 2, 6),
            "num_layers": trial.suggest_int("num_layers", 1, 5),
            "pool_size": trial.suggest_int("pool_size", 2, 5),
            "stride": trial.suggest_int("stride", 1, 4)
        }

    trainer = ModelActioner(train_data, test_data, device, model_type)

    val_loss = trainer.train_validate(config, trial)

    return val_loss

In [375]:
study_name = "Vanilla-LSTM-Tunner"
storage_url = "sqlite:///db.sqlite3"

optuna.delete_study(study_name=study_name, storage= storage_url)

study = optuna.create_study(direction='minimize', 
                            storage=storage_url, 
                            sampler=TPESampler(),
                            pruner=optuna.pruners.HyperbandPruner(
                            min_resource=30,  
                            max_resource=150,  
                            reduction_factor=3,  
                            ),
                            study_name=study_name,
                            load_if_exists=False)

pbar = tqdm(total=25, desc='Optimizing', unit='trial')

def callback(study, trial):
    # Update the progress bar
    pbar.update(1)
    # Optionally, you can include additional information in the progress bar
    pbar.set_postfix_str(f"Best Value: {study.best_value:.4f}")


study.optimize(objective, n_trials=25, callbacks=[callback])
pbar.close()

# Best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('Value:', trial.value)
print('Params:')
for key, value in trial.params.items():
    print(f'{key}: {value}')

[I 2024-01-27 01:09:27,508] A new study created in RDB with name: Vanilla-LSTM-Tunner


Optimizing:   0%|          | 0/25 [00:00<?, ?trial/s]

Starting fold 1:
epochs 1/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.087007609166597, val_loss: 1.0812877554642526
epochs 2/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.087238543911984, val_loss: 1.0810697819057264
epochs 3/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.0865755419982106, val_loss: 1.080854374484012
epochs 4/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.0862810047049272, val_loss: 1.080641285996688
epochs 5/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.086279156333522, val_loss: 1.0804289228037784
epochs 6/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.0860497286445217, val_loss: 1.080216767913417
epochs 7/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.086237879803306, val_loss: 1.0800046130230552
epochs 8/108
Current Learning Rate: 9.69243553136279e-06
train_loss: 1.0857690246481644, val_loss: 1.0797940956918817
epochs 9/108
Current Learning Rate: 9.69243553

[I 2024-01-27 01:10:08,848] Trial 0 finished with value: 1.0732808287764153 and parameters: {'batch_size': 92, 'epochs': 108, 'hidden_size': 69, 'learning_rate': 9.69243553136279e-06, 'dropout_prob': 0.11252149882858338, 'weight_decay': 1.5406397074921667e-05, 'lr_step_size': 97, 'gamma': 0.29493266281464603, 'conv_channels': 75, 'kernel_size': 6, 'num_layers': 2, 'pool_size': 5, 'stride': 3}. Best is trial 0 with value: 1.0732808287764153.


Current Learning Rate: 2.8586158204241164e-06
train_loss: 1.0327717738402518, val_loss: 1.1066702089811626
Mean validation loss: 1.0732808287764153
Starting fold 1:
epochs 1/81
Current Learning Rate: 0.02232859040835427
train_loss: 1.1484769582748413, val_loss: 0.9975131452083588
epochs 2/81
Current Learning Rate: 0.02232859040835427
train_loss: 0.9878760874271393, val_loss: 0.9965087473392487
epochs 3/81
Current Learning Rate: 0.02232859040835427
train_loss: 0.9996513426303864, val_loss: 1.0050821006298065
epochs 4/81
Current Learning Rate: 0.02232859040835427
train_loss: 1.0042048692703247, val_loss: 1.004830926656723
epochs 5/81
Current Learning Rate: 0.02232859040835427
train_loss: 1.0096585750579834, val_loss: 1.0002799332141876
epochs 6/81
Current Learning Rate: 0.02232859040835427
train_loss: 1.006783127784729, val_loss: 0.9964340329170227
epochs 7/81
Current Learning Rate: 0.02232859040835427
train_loss: 0.9986208975315094, val_loss: 0.9949827194213867
epochs 8/81
Current Learn

...

Current Learning Rate: 0.07805253881073754
train_loss: 0.9960882916262276, val_loss: 1.112464672954459
epochs 16/76
Starting fold 1:
epochs 1/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.096008475203263, val_loss: 1.0285240637628656
epochs 2/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.045541094478808, val_loss: 0.9968785762786865
epochs 3/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.0005874997691104, val_loss: 0.9884346974523444
epochs 4/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.0003988993795294, val_loss: 0.9881737483175177
epochs 5/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.0090683253187882, val_loss: 0.9902554135573538
epochs 6/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.0040628031680459, val_loss: 0.9917647938979299
epochs 7/97
Current Learning Rate: 0.0056702958366915916
train_loss: 1.0032682299613953, val_loss: 0.9931007623672485
epochs 8/97
Current Learning Rate: 0.005670

[I 2024-01-27 01:14:17,517] Trial 23 pruned. 


Starting fold 1:
epochs 1/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1522886753082275, val_loss: 1.1521728873252868
epochs 2/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.152109146118164, val_loss: 1.1503965258598328
epochs 3/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1473880290985108, val_loss: 1.148645031452179
epochs 4/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1484215021133424, val_loss: 1.146907424926758
epochs 5/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1429500222206115, val_loss: 1.1451788425445557
epochs 6/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1450612664222717, val_loss: 1.143454670906067
epochs 7/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1409581899642944, val_loss: 1.1417362093925476
epochs 8/111
Current Learning Rate: 0.00013181339015927505
train_loss: 1.1381697177886962, val_loss: 1.1400108933448792
epochs 9/111
Current Learni

[I 2024-01-27 01:14:19,675] Trial 24 pruned. 


Number of finished trials: 25
Best trial:
Value: 1.0725992555994734
Params:
batch_size: 39
epochs: 95
hidden_size: 28
learning_rate: 0.0003278964463346113
dropout_prob: 0.14145831654644206
weight_decay: 0.0009921223812580742
lr_step_size: 37
gamma: 0.1093779113711891
conv_channels: 61
kernel_size: 5
num_layers: 5
pool_size: 3
stride: 4


In [376]:
trainer = ModelActioner(train_data, test_data, device, model_type)
model = trainer.train(trial.params)

epochs 1/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.089576555239527
epochs 2/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0769140507045545
epochs 3/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0646134042426159
epochs 4/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.057625372786271
epochs 5/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0531212317316156
epochs 6/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0531311609243093
epochs 7/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.052690472885182
epochs 8/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0531987240439966
epochs 9/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0541629221878555
epochs 10/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0534992650935524
epochs 11/95
Current Learning Rate: 0.0003278964463346113
train_loss: 1.0529172360897063
epochs 12/95
Current Learning Rat

In [377]:
preds = trainer.test(trial.params)
y_true = y_test[time_step:]

accuracy = accuracy_score(y_true, preds)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Precision
precision = precision_score(y_true, preds, average='weighted', zero_division=0)
print(f'Precision: {precision:.4f}')

# Recall
recall = recall_score(y_true, preds, average='weighted')
print(f'Recall: {recall:.4f}')

# F1 Score
f1 = f1_score(y_true, preds, average='weighted')
print(f'F1 Score: {f1:.4f}')

test_loss: 0.9543475097134
Accuracy: 63.89%
Precision: 0.4082
Recall: 0.6389
F1 Score: 0.4981


In [378]:
unique_elements = set(preds)
print(unique_elements)

{0}


In [379]:
y_true = y_true.reset_index(drop=True)
#y_true = y_true.drop(columns=["index","level_0"])
X_value = X_test_df.iloc[43:,-1]
X_value = X_value.reset_index()
preds_values = [x[0] for x in preds]
X_value["Preds"] = pd.Series(preds_values)
X_value["Next Day"] = y_true
X_value = X_value.drop(columns=["index"])
X_value["Diff Preds"] = (X_value["Preds"] - X_value["Adj Close"])
X_value["Diff Next Day"] = (X_value["Next Day"] - X_value["Adj Close"])
X_value = X_value.dropna()
X_value

IndexError: invalid index to scalar variable.

In [ ]:
def assign_label(diff):
    if diff > 0.0:
        return 1  # Buy
    elif diff < -0.01:
        return 0  # Sell
    else:
        return 2  # Hold/Other

In [ ]:
# Apply the function to assign labels
X_value['Signal Preds'] = X_value['Diff Preds'].apply(assign_label)
X_value['Signal True'] = X_value['Diff Next Day'].apply(assign_label)

# Dropping the intermediate difference columns if they are no longer needed
X_value = X_value.drop(columns=['Diff Preds', 'Diff Next Day'])

X_value

,Adj Close,Preds,Next Day,Signal Preds,Signal True
0,0.706173,-7.781130e-40,0.728267,0,1
1,0.728267,-7.781130e-40,0.737950,0,1
2,0.737950,-7.781130e-40,0.733381,0,2
3,0.733381,-7.781130e-40,0.747701,0,1
4,0.747701,-7.781130e-40,0.761134,0,1
...,...,...,...,...,...
247,1.025207,-7.781130e-40,1.018693,0,2
248,1.018693,-7.781130e-40,1.059493,0,1
249,1.059493,-7.781130e-40,1.079584,0,1
250,1.079584,-7.781130e-40,1.095561,0,1


In [ ]:
accuracy = accuracy_score(X_value["Signal True"], X_value["Signal Preds"])
print(f'Accuracy: {accuracy * 100:.2f}%')

# Precision
precision = precision_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'Precision: {precision:.4f}')

# Recall
recall = recall_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'Recall: {recall:.4f}')

# F1 Score
f1 = f1_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'F1 Score: {f1:.4f}')


Accuracy: 19.44%
Precision: 0.0378
Recall: 0.1944
F1 Score: 0.0633


/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [ ]:
y_true

0      0.728267
1      0.737950
2      0.733381
3      0.747701
4      0.761134
         ...   
247    1.018693
248    1.059493
249    1.079584
250    1.095561
251    1.104406
Name: Adj Close, Length: 252, dtype: float64

In [ ]:
X_value["Signal Preds"]

0      0
1      0
2      0
3      0
4      0
      ..
247    0
248    0
249    0
250    0
251    0
Name: Signal Preds, Length: 252, dtype: int64

In [ ]:
signals = pd.DataFrame(X_value["Signal Preds"].values, columns=['Signal'])
signals

,Signal
0,0
1,0
2,0
3,0
4,0
...,...
247,0
248,0
249,0
250,0


In [ ]:
signals["Signal"].value_counts()

Signal
0    252
Name: count, dtype: int64

In [ ]:
signals["Date"] = date_test
signals

,Signal,Date
0,0,2023-01-23
1,0,2023-01-24
2,0,2023-01-25
3,0,2023-01-26
4,0,2023-01-27
...,...,...
247,0,2024-01-17
248,0,2024-01-18
249,0,2024-01-19
250,0,2024-01-22


In [ ]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325668
1,141.737762
2,141.071487
3,143.159805
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [ ]:
date_test["Date"] = date_test["Date"].dt.strftime('%Y-%m-%d')
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(date_test).flatten(), y=stock_data["Adj Close"].iloc[test_index:], mode='lines', name='TSLA Stock Price'))

# Buy and sell signals
buy_signals = signals[signals['Signal'] == 1]
sell_signals = signals[signals['Signal'] == 0]

# Ensure that buy and sell prices are aligned with the signals by matching on the 'Date' column
buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]

# Plot buy signals
fig.add_trace(go.Scatter(
    x=buy_signals['Date'], 
    y=buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
fig.add_trace(go.Scatter(
    x=sell_signals['Date'], 
    y=sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update layout
fig.update_layout(
    title='Stock Price with Buy and Sell Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height = 700,
    width=1280
)

# Show the plot
pyo.iplot(fig)

/home/arda/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [ ]:
price_signal = stock_data[stock_data['Date'].isin(signals['Date'])]["Adj Close"]
price_signal = price_signal.reset_index()
price_signal = price_signal.drop(columns=["index"])
result = pd.concat([price_signal,signals], axis=1)
result

,Adj Close,Signal,Date
0,140.325668,0,2023-01-23
1,141.737762,0,2023-01-24
2,141.071487,0,2023-01-25
3,143.159805,0,2023-01-26
4,145.118851,0,2023-01-27
...,...,...,...
247,182.679993,0,2024-01-17
248,188.630005,0,2024-01-18
249,191.559998,0,2024-01-19
250,193.889999,0,2024-01-22


In [ ]:
price_signal

,Adj Close
0,140.325668
1,141.737762
2,141.071487
3,143.159805
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [ ]:
sell_signals

,Signal,Date
0,0,2023-01-23
1,0,2023-01-24
2,0,2023-01-25
3,0,2023-01-26
4,0,2023-01-27
...,...,...
247,0,2024-01-17
248,0,2024-01-18
249,0,2024-01-19
250,0,2024-01-22


In [ ]:
buy_prices

Series([], Name: Adj Close, dtype: float64)